# 01 - AXI DMA loopback (board bring-up)

Sends data from DDR through the FPGA (`AXI DMA MM2S -> FIFO -> AXI DMA S2MM`)
and back, and checks every word. Works on PYNQ 2.5 and 3.x.

Copy `dma_loopback.bit` and `dma_loopback.hwh` into the same folder as this notebook.

In [ ]:
import os, time
import numpy as np
import pynq
from pynq import Overlay, allocate
print("PYNQ", pynq.__version__)

ol = Overlay(os.path.abspath("dma_loopback.bit"))
print("IP blocks:", list(ol.ip_dict.keys()))
dma = ol.axi_dma_0

In [ ]:
def loopback(src):
    dst = allocate(shape=src.shape, dtype=src.dtype)
    dst[:] = 0
    dma.recvchannel.transfer(dst)   # arm the receiver first
    dma.sendchannel.transfer(src)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    out = np.array(dst)
    dst.freebuffer()
    return out

# Test 1: counter and pseudo-random 32-bit words
rng = np.random.RandomState(0)
for name, data in [("counter", np.arange(1024, dtype=np.uint32)),
                   ("random",  rng.randint(0, 2**32, size=4096, dtype=np.uint64).astype(np.uint32))]:
    src = allocate(shape=data.shape, dtype=np.uint32)
    src[:] = data
    out = loopback(src)
    print("%-8s %5d words: %s" % (name, data.size, "PASS" if np.array_equal(out, data) else "FAIL"))
    src.freebuffer()

In [ ]:
# Test 2: one DOA frame in the accelerator's stream format
# N = 8 elements x M = 128 snapshots, one complex sample per 32-bit word:
# [15:0] = Re (int16), [31:16] = Im (int16), snapshot-major order.
N, M = 8, 128
x = (rng.randn(N, M) + 1j * rng.randn(N, M))
x /= 1.001 * max(np.abs(x.real).max(), np.abs(x.imag).max())
re = np.round(x.real * 2**15).astype(np.int16)
im = np.round(x.imag * 2**15).astype(np.int16)
words = (re.T.astype(np.uint16).astype(np.uint32)
         | (im.T.astype(np.uint16).astype(np.uint32) << 16)).ravel()

src = allocate(shape=words.shape, dtype=np.uint32)
src[:] = words
out = loopback(src)
re_back = (out & 0xFFFF).astype(np.uint16).astype(np.int16).reshape(M, N).T
im_back = (out >> 16).astype(np.uint16).astype(np.int16).reshape(M, N).T
print("frame (%d words): %s" % (words.size,
      "PASS" if np.array_equal(re_back, re) and np.array_equal(im_back, im) else "FAIL"))

In [ ]:
# Test 3: host-observed transfer time for one frame (median of 200)
times = []
for _ in range(200):
    dst = allocate(shape=words.shape, dtype=np.uint32)
    t0 = time.perf_counter()
    dma.recvchannel.transfer(dst)
    dma.sendchannel.transfer(src)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    times.append(time.perf_counter() - t0)
    dst.freebuffer()
t = np.array(times) * 1e6
print("1024-word round trip: median %.1f us, min %.1f us, max %.1f us" % (np.median(t), t.min(), t.max()))
print("(host-observed, includes Python call overhead)")
src.freebuffer()